# 02 - LoRA on SAM 3 (team stack)
The LoRA adaptation arm on the native `sam3` package, the shared team stack
(`Sam3MedSeg`). Build the model, verify the injection targets, and verify the
trainable-param counts of `finetune_mode="lora"`.

History (in git, not the working tree): the SAM 1 prototype lives in this
notebook's earlier versions (147K/93.7M, 0.157%); the HF-transformers SAM 3
verification was notebook 03_Sam3_explore.ipynb, removed after the team
standardized on the native stack (see commit 6d38175).

**Kernel: select the `sam3-native` env** (the only one with the `sam3` package).

In [1]:
#autoreload, so edits to .py files show without restarting
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.insert(0, "../src")   #repo convention so `medseg` imports
print(sys.executable)          #sanity check: should be the sam3-native env

/Users/nikhilclaude/anaconda3/envs/sam3-native/bin/python


## 1. Unit tests: LoRALayer / LinearWithLoRA / LoRA_qkv slice isolation

In [3]:
!pytest ../tests/test_lora.py -v

============================= test session starts ==============================
platform darwin -- Python 3.12.13, pytest-9.1.1, pluggy-1.6.0 -- /Users/nikhilclaude/anaconda3/envs/sam3-native/bin/python3.12
cachedir: .pytest_cache
rootdir: /Users/nikhilclaude/Documents/Projects/DeepLearning/group_project
plugins: anyio-4.14.2
collected 5 items                                                              

../tests/test_lora.py::test_output_shape PASSED                          [ 20%]
../tests/test_lora.py::test_zero_at_init PASSED                          [ 40%]
../tests/test_lora.py::test_param_count PASSED                           [ 60%]
../tests/test_lora.py::test_linear_with_lora PASSED                      [ 80%]
../tests/test_lora.py::test_lora_qkv_slices PASSED                       [100%]

============================== 5 passed in 0.73s ===============================


## 2. Build the native SAM 3 image model
Loads the gated weights from the HF cache (`load_from_HF=True` default), builds
on CPU (`device='cpu'` default). Takes a minute or two. This is the exact call
inside `Sam3Backbone.__init__`.

In [4]:
from sam3.model_builder import build_sam3_image_model

model = build_sam3_image_model()
total = sum(p.numel() for p in model.parameters())
print(f"{type(model).__name__}: {total:,} total params")

/Users/nikhilclaude/Documents/Projects/DeepLearning/group_project/vendor/sam3-repo/sam3/model_builder.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/nikhilclaude/Documents/Projects/DeepLearning/group_project/vendor/sam3-repo/sam3/model/sam3_video_inference.py:800: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.bfloat16)
/Users/nikhilclaude/Documents/Projects/DeepLearning/group_project/vendor/sam3-repo/sam3/model/sam3_video_inference.py:908: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.bfloat16)
/Users/nikhilclaude/anaconda3/envs/sam3-native/lib/python3.12/site-packages/timm/models/layers/__in

Sam3Image: 840,509,750 total params


## 3. LoRA injection targets: 32 fused-qkv attention layers
Native vitdet fuses Q,K,V into ONE `Linear(1024, 3072)` per block (unlike the
HF edition's separate q_proj/v_proj), so we wrap with `LoRA_qkv`, not
`LinearWithLoRA`. Expect 32 blocks, each
`Linear(in_features=1024, out_features=3072, bias=True)`.

In [5]:
blocks = model.backbone.vision_backbone.trunk.blocks
print(f"{len(blocks)} blocks")
print(blocks[0].attn.qkv)

32 blocks
Linear(in_features=1024, out_features=3072, bias=True)


## 4. The LoRA arm: `Sam3MedSeg(finetune_mode="lora")`
Freeze the 455M-param backbone, inject adapters into all 32 qkv layers, train
adapters + FPN seg head. Expected (verified 2026-07-20):

```
1,066,113 / 455,105,921 trainable  (0.2343%)
  = 524,288 adapters (32 x 2 x 2*4*1024)  +  541,825 FpnSegHead
```

Any other number means the injection state is wrong (e.g. head swept into the
freeze, or a double injection; see docs/lora.md).

In [6]:
from medseg.models.sam3_medseg import Sam3MedSeg

m = Sam3MedSeg(finetune_mode="lora")
trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
total = sum(p.numel() for p in m.parameters())
print(f"{trainable:,} / {total:,} trainable  ({100*trainable/total:.4f}%)")

1,066,113 / 455,105,921 trainable  (0.2343%)


## 5. Smoke training run
Runs `scripts/train_lora.py`: tiny BUSI subset (8 images, 2 epochs, batch 1)
through the LoRA arm, using the shared Trainer (AdamW, BCE+Dice, Dice/NSD val).
Config: `configs/experiment/sam3_lora_busi.yaml`.

Notes:
- The `cd ..` matters: the script resolves `src/` and `configs/` relative to
  the repo root, and a notebook's working directory is `notebooks/`.
- First run downloads BUSI from Zenodo into `~/.medsegbench`.
- Success = train_loss drops between epochs and `best.pt` appears in
  `checkpoints/sam3_lora_busi_test/`. Slow on CPU; MPS is untested territory,
  if it errors switch the script's device to "cpu" and rerun.

In [ ]:
!cd .. && python scripts/train_lora.py

## 6. Visualize predictions: untrained vs trained

See what the 2-epoch smoke run actually learned, on a held-out (val) BUSI image.

Two important caveats on the "untrained" baseline:
- A fresh `Sam3MedSeg(finetune_mode="lora")` has LoRA adapters at B=0 (they do
  nothing, so the backbone behaves exactly as pretrained SAM 3) AND a randomly
  initialized seg head. So "untrained" = frozen SAM 3 features through a random
  head, which produces near-noise. It is NOT zero-shot SAM 3 (that promptable
  pipeline was removed from the team stack).
- So this compares **0 training steps vs 2 epochs on 8 images**. It shows what a
  tiny amount of training bought, not a foundation-model-vs-adapted comparison.

We build one model (that is the untrained baseline), predict, then load the
checkpoint into the same object and predict again (one model in memory).

In [ ]:
import sys; sys.path.insert(0, "../src")
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from medseg.data.medseg_data import DataConfig, build_dataloaders
from medseg.models import Sam3MedSeg
from medseg.utils.device import get_device

# --- what to look at ---
device = get_device()                                    # mps on this Mac
CKPT = "../checkpoints/sam3_lora_busi_test/best.pt"      # the 2-epoch smoke checkpoint
SPLIT = "test"   # "test" = never seen at all; "val" = unseen by training but picked the checkpoint
IDX = 0          # which image within the split (order is deterministic, so this is reproducible)

# Build the SAME dataloaders the training run used, then reach past the loader
# to the underlying dataset so we can grab one sample by index. Each sample is a
# dict; we only need the image and its ground-truth mask here.
cfg = DataConfig.from_yaml("../configs/experiment/sam3_lora_busi.yaml")
ds = build_dataloaders(cfg)[SPLIT].dataset
sample = ds[IDX]
image = sample["image"]          # (3,256,256) float in [0,1]  (3 channels: BUSI is grayscale copied to RGB)
gt = sample["mask"][0]           # (256,256)  float {0,1}       ([0] drops the channel axis)
print(f"{SPLIT}[{IDX}] | image {tuple(image.shape)} | lesion pixels in GT: {int(gt.sum().item())}")

In [ ]:
@torch.no_grad()   # inference only, so don't track gradients (faster, less memory)
def predict_prob(model, image):
    """Run one image through the model and return a 256x256 lesion-probability map.

    The model takes (B,3,H,W) and outputs (B,1,1008,1008) logits (it resizes to
    1008 internally). We add a batch dim, sigmoid the logits into 0..1
    probabilities, then resize back down to 256 so it lines up with the image
    and ground-truth mask for display and scoring.
    """
    model.eval()
    logits = model(image.unsqueeze(0).to(device))         # (1,1,1008,1008)
    prob = torch.sigmoid(logits)                          # logits -> probability in [0,1]
    prob = F.interpolate(prob, size=image.shape[-2:],     # 1008 -> 256
                         mode="bilinear", align_corners=False)
    return prob[0, 0].cpu()                               # drop batch+channel -> (256,256)

def dice_of(prob, gt, thr=0.5):
    """Dice overlap between the thresholded prediction and the GT mask (1 = perfect)."""
    pred = (prob > thr).float()                           # probability map -> binary mask
    inter = (pred * gt).sum()                             # pixels both call lesion
    return (2 * inter / (pred.sum() + gt.sum() + 1e-6)).item()

# A fresh model IS the untrained baseline: LoRA adapters start at B=0 (zero effect,
# backbone == pretrained SAM 3) and the seg head is randomly initialized. So this
# predicts with a random head -> expect noise. (Building it loads SAM 3: ~1-2 min.)
model = Sam3MedSeg(finetune_mode="lora").to(device)
prob_untrained = predict_prob(model, image)
print(f"untrained dice: {dice_of(prob_untrained, gt):.4f}")

In [ ]:
# Load the 2-epoch checkpoint into the SAME model object. This overwrites the
# random head and zero adapters with the trained ones, turning the baseline model
# above into the trained model (no second SAM 3 build, so it stays light on memory).
# best.pt is a dict: {"model": state_dict, "epoch": int, "dice": float}.
ckpt = torch.load(CKPT, map_location=device)
model.load_state_dict(ckpt["model"])
prob_trained = predict_prob(model, image)
print(f"checkpoint epoch {ckpt['epoch']} (val_dice {ckpt['dice']:.4f}) | "
      f"this image trained dice: {dice_of(prob_trained, gt):.4f}")

In [ ]:
# Four panels side by side for the single image (IDX) chosen above:
#   1) the raw ultrasound
#   2) the same image with the ground-truth lesion painted red
#   3) the UNTRAINED probability map (random head -> noise)
#   4) the TRAINED probability map (should concentrate on the lesion)
# viridis colormap: dark blue = low probability, yellow = high.
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
img_np = image.permute(1, 2, 0).numpy()   # (3,256,256) -> (256,256,3) for imshow, values in [0,1]

axes[0].imshow(img_np)
axes[0].set_title("ultrasound image")

axes[1].imshow(img_np)
axes[1].imshow(gt, cmap="Reds", alpha=0.5)          # overlay GT mask, semi-transparent
axes[1].set_title("ground-truth lesion")

axes[2].imshow(prob_untrained, cmap="viridis", vmin=0, vmax=1)
axes[2].set_title(f"untrained (0 steps)\ndice {dice_of(prob_untrained, gt):.3f}")

axes[3].imshow(prob_trained, cmap="viridis", vmin=0, vmax=1)
axes[3].set_title(f"trained (2 epochs)\ndice {dice_of(prob_trained, gt):.3f}")

for a in axes:
    a.axis("off")
plt.tight_layout(); plt.show()

### Several images at once
The trained model (already loaded above) on a few images from the same split, to
see the range: it does better on large, clear lesions than small/low-contrast ones.

In [ ]:
# One row per image: [ image | ground truth | untrained pred | trained pred ].
# We keep BOTH the untrained baseline and the trained prediction on every row so
# the before/after is visible for each image, not just the single one above.
#
# `model` currently holds the TRAINED weights (loaded above). To also get the
# untrained baseline we rebuild a fresh model once (adapters at B=0 + random head).
# Building SAM 3 takes ~1-2 min; we do it a single time and reuse it for all rows.
INDICES = [0, 1, 2]

baseline = Sam3MedSeg(finetune_mode="lora").to(device)   # fresh = untrained baseline

rows = len(INDICES)
fig, axes = plt.subplots(rows, 4, figsize=(16, 4 * rows))
for row, idx in enumerate(INDICES):
    s = ds[idx]
    img, m = s["image"], s["mask"][0]
    prob_u = predict_prob(baseline, img)            # untrained (random head)
    prob_t = predict_prob(model, img)               # trained (checkpoint loaded above)
    img_np = img.permute(1, 2, 0).numpy()

    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title(f"{SPLIT}[{idx}] image")
    axes[row, 1].imshow(img_np)
    axes[row, 1].imshow(m, cmap="Reds", alpha=0.5)
    axes[row, 1].set_title("ground truth")
    axes[row, 2].imshow(prob_u, cmap="viridis", vmin=0, vmax=1)
    axes[row, 2].set_title(f"untrained\ndice {dice_of(prob_u, m):.3f}")
    axes[row, 3].imshow(prob_t, cmap="viridis", vmin=0, vmax=1)
    axes[row, 3].set_title(f"trained\ndice {dice_of(prob_t, m):.3f}")
    for a in axes[row]:
        a.axis("off")
plt.tight_layout(); plt.show()

### Cross-dataset test: the BUSI-trained model on Kvasir
Same trained model, but run on Kvasir (RGB endoscopy polyps) instead of BUSI
(grayscale ultrasound lesions). This is an out-of-distribution / domain-shift
test: the frozen SAM 3 backbone is general, but the trained head + adapters
learned the BUSI task specifically, so expect poor, diffuse predictions. This is
why the project trains one model per dataset. (First run downloads Kvasir.)

In [ ]:
# Load Kvasir (a different dataset) via its data config. `model` is the BUSI-trained
# one, and `baseline` is the untrained one, both built above. First run downloads
# Kvasir from Zenodo. Columns: image | GT polyp | untrained | BUSI-trained.
kvasir_cfg = DataConfig.from_yaml("../configs/data/kvasir.yaml")
kv_ds = build_dataloaders(kvasir_cfg)["test"].dataset
print("Kvasir test images:", len(kv_ds))

KV_INDICES = [0, 1, 2]
rows = len(KV_INDICES)
fig, axes = plt.subplots(rows, 4, figsize=(16, 4 * rows))
for row, idx in enumerate(KV_INDICES):
    s = kv_ds[idx]
    img, m = s["image"], s["mask"][0]
    prob_u = predict_prob(baseline, img)            # untrained (random head)
    prob_t = predict_prob(model, img)               # BUSI-trained model on a Kvasir image
    img_np = img.permute(1, 2, 0).numpy()

    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title(f"kvasir[{idx}] image")
    axes[row, 1].imshow(img_np)
    axes[row, 1].imshow(m, cmap="Reds", alpha=0.5)
    axes[row, 1].set_title("ground truth (polyp)")
    axes[row, 2].imshow(prob_u, cmap="viridis", vmin=0, vmax=1)
    axes[row, 2].set_title(f"untrained\ndice {dice_of(prob_u, m):.3f}")
    axes[row, 3].imshow(prob_t, cmap="viridis", vmin=0, vmax=1)
    axes[row, 3].set_title(f"BUSI-trained\ndice {dice_of(prob_t, m):.3f}")
    for a in axes[row]:
        a.axis("off")
plt.tight_layout(); plt.show()